# Actividad 1

Ivan Condado Narvaez - 202320561

## A partir de la base de datos Titanic, se realizará una actividad en la cual:
  1. Analizar si la muestra es suficiente o no para una predicción si una persona sobrevive o no.
  2. ¿Se puede usar un número menor de instancias que contiene la muestra inicial sin perder representatividad?
  3. ¿Qué tamaño de conjuntos train y test se debe de usar sin perder representatividad de cada partición? ¿Qué porcentaje de división se deberá emplear?
Se deben de argumentar las respuestas

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

from sklearn.model_selection import GridSearchCV

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

# 1. Analizar si la muestra es suficiente o no para una prediccion si una persona sobrevive o no

Lo primero que tenemos que hacer para determinar si estas muestras son suficientes es sacar cuantos datos tenemos. Esta datasheet del titanic tiene muchos datos faltantes en la columna de "Cabin", por lo que a pesar de ser util, debemos eliminarla.

In [8]:

# Cargar el archivo CSV
data = pd.read_csv("titanic_dataset.csv")

# Convertimos los valores de String a int para que el modelo se entrene correctamente
data['Sex'] = data['Sex'].astype('category').cat.codes # male = 1, female = 0
data['Embarked'] = data['Embarked'].astype('category').cat.codes # C = 0, Q = 1, S = 2

demostracion = data.drop(columns=["PassengerId", "Name", "Ticket"])

data = data.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
# Cabin es util, pero tiene demasiados elementos vacios (529 incluso despues de quitar las demas columnas)
# Ademas, la mayoria de los elementos que SI tienen cabin, son de clase alta, pues los que no se registraron
# fueron en su inmensa mayoria, los de clase baja


data = data.dropna()
demostracion = demostracion.dropna()

# Separar características (X) y etiquetas (y)
X = data.drop(columns=["Survived"])
y = data["Survived"]

# Pclass = clase economica  (1 alta, 2 media, 3 baja)
# SibSp = Hermanos / matrimonio a bordo
# Parch = Padres / hijos a bordo
# Ticket = numero de billete
# Fare = tarifa pagada por el billete
# Cabin = numero de camarote
# Embarked = puerto donde embarco el pasajero (c = cherburgo, q = queenstown, s = southampton

display(data)
display(demostracion)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,1,22.0,1,0,7.2500,2
1,1,1,0,38.0,1,0,71.2833,0
2,1,3,0,26.0,0,0,7.9250,2
3,1,1,0,35.0,1,0,53.1000,2
4,0,3,1,35.0,0,0,8.0500,2
...,...,...,...,...,...,...,...,...
885,0,3,0,39.0,0,5,29.1250,1
886,0,2,1,27.0,0,0,13.0000,2
887,1,1,0,19.0,0,0,30.0000,2
889,1,1,1,26.0,0,0,30.0000,0


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
1,1,1,0,38.0,1,0,71.2833,C85,0
3,1,1,0,35.0,1,0,53.1000,C123,2
6,0,1,1,54.0,0,0,51.8625,E46,2
10,1,3,0,4.0,1,1,16.7000,G6,2
11,1,1,0,58.0,0,0,26.5500,C103,2
...,...,...,...,...,...,...,...,...,...
871,1,1,0,47.0,1,1,52.5542,D35,2
872,0,1,1,33.0,0,0,5.0000,B51 B53 B55,2
879,1,1,0,56.0,0,1,83.1583,C50,0
887,1,1,0,19.0,0,0,30.0000,B42,2


Como podemos ver, tenemos 714 datos si borramos la columna "Cabin", mientras que solo 185 si la dejamos. Incluso 714 datos son pocos para entrenar un modelo, pero sigue siendo mucho mejor que solo 185

In [25]:
# Preparamos los datos para entrenar nuestro modelo

# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Normalizar los datos (opcional pero recomendable para RNA)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Datos preparados para ser usados en una RNA con scikit-learn.")

Datos preparados para ser usados en una RNA con scikit-learn.


Con los datos listos, usaremos grid_search para buscar el mejor conjunto de parametros para entrenar nuestro modelo. Por esta ocasion solo vamos a buscar hiperparametros diferentes para "hidden_layer_sizes" y "alpha"

In [10]:

param_grid = {
    "hidden_layer_sizes": [(8,), (16,), (32,), (4,4), (8,8), (16,16)],
    "alpha": [0.00001, 0.0001, 0.001],
    "max_iter": [5000]
}


# Instanciar el modelo base
rna = MLPClassifier(random_state=42)

# Aplicar GridSearchCV para encontrar la mejor combinacion de hiperparametros
grid_search = GridSearchCV(rna, param_grid, cv=5, scoring="accuracy", n_jobs=1)
grid_search.fit(X_train, y_train)

# Mejor configuracion encontrada
print(f"Mejores hiperparametros encontrados: {grid_search.best_params_}")
print(f"Precision en datos de prueba: {grid_search.best_estimator_.score(X_test, y_test)}")


Mejores hiperparametros encontrados: {'alpha': 0.0001, 'hidden_layer_sizes': (8,), 'max_iter': 5000}
Precision en datos de prueba: 0.8041958041958042


grid_search nos arroja como resultado que los mejores parametros son:
- alpha: 0.0001
- hidden_layer_sizes: (8,)
para 5000 epocas, estos son los hiperparametros que vamos a usar para entrenar nuestro modelo

In [11]:
clf = MLPClassifier(hidden_layer_sizes=(8,), max_iter=5000, alpha=0.0001, random_state=42)
clf.fit(X_train, y_train)

# Evaluar el modelo
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Precisión del modelo: {accuracy:.2f}")

Precisión del modelo: 0.80


Lo que calcula accuracy_score es $\frac{Predicciones\ correctas}{Total\ de\ predicciones}$, lo que quiere decir que por cada 100 predicciones que hace nuestro modelo, 80 de ellas son correctas

**Respuesta 1:** Aunque no tiene una precision muy elevada, el entrenamiento demuestra que el modelo es capaz de predecir si una persona sobrevive o no el 80% de las veces, lo cual, a pesar de no ser tan confiable como nos gustaria, puede considerarse un exito teniendo en cuenta la reducida cantidad de datos que tenemos

# 2. ¿Se puede usar un número menor de instancias que contiene la muestra inicial sin perder representatividad?

Para esta pregunta, podemos volver a utilizar  tabla de demostracion que usamos al principio, pues es una tabla con un numero menor de instancias en la muestra inicial

In [13]:
# Removeremos la columna Cabin para que ambas tablas esten en igualdad de condiciones
demostracion = demostracion.drop(columns=["Cabin"])

# Separar características (X) y etiquetas (y)
X_2 = demostracion.drop(columns=["Survived"])
y_2 = demostracion["Survived"]

# Dividir los datos en conjuntos de entrenamiento y prueba
X_2_train, X_2_test, y_2_train, y_2_test = train_test_split(X_2, y_2, test_size=0.20, random_state=42)

# Normalizar los datos (opcional pero recomendable para RNA)
scaler = StandardScaler()
X_2_train = scaler.fit_transform(X_2_train)
X_2_test = scaler.transform(X_2_test)

print("Datos preparados para ser usados en una RNA con scikit-learn.")

Datos preparados para ser usados en una RNA con scikit-learn.


Una vez los datos esten listos, volveremos a repetir la parte de buscar hiperparametros y entrenar nuestro modelo

In [15]:

param_grid = {
    "hidden_layer_sizes": [(8,), (16,), (32,), (4,4), (8,8), (16,16)],
    "alpha": [0.00001, 0.0001, 0.001],
    "max_iter": [5000]
}


# Instanciar el modelo base
rna = MLPClassifier(random_state=42)

# Aplicar GridSearchCV para encontrar la mejor combinacion de hiperparametros
grid_search = GridSearchCV(rna, param_grid, cv=5, scoring="accuracy", n_jobs=1)
grid_search.fit(X_2_train, y_2_train)

# Mejor configuracion encontrada
print(f"Mejores hiperparametros encontrados: {grid_search.best_params_}")
print(f"Precision en datos de prueba: {grid_search.best_estimator_.score(X_2_test, y_2_test)}")


Mejores hiperparametros encontrados: {'alpha': 0.001, 'hidden_layer_sizes': (4, 4), 'max_iter': 5000}
Precision en datos de prueba: 0.6756756756756757


grid_search nos arroja como resultado que los mejores parametros son:
- alpha: 0.001
- hidden_layer_sizes: (4,4)
para 5000 epocas, estos son los hiperparametros que vamos a usar para entrenar nuestro nuevo modelo

In [20]:
clf = MLPClassifier(hidden_layer_sizes=(4,4), max_iter=5000, alpha=0.001, random_state=42)
clf.fit(X_2_train, y_2_train)

# Evaluar el modelo
y_2_pred = clf.predict(X_2_test)
accuracy = accuracy_score(y_2_test, y_2_pred)

print(f"Precisión del modelo: {accuracy:.2f}")

Precisión del modelo: 0.68


**Respuesta 2:** No, no podemos usar un numero menor de instancias pues la precision cae en picado. 714 datos son de por si pocos, por lo que si reducimos aun mas esta cantidad (185), nuestro modelo no va a tener ni de cerca datos suficientes para entrenar y como consecuencia, va a tener una precision bajisima

# 3. ¿Qué tamaño de conjuntos train y test se debe de usar sin perder representatividad de cada partición? ¿Qué porcentaje de división se deberá emplear?

**Respuesta:** Al ser un dataset pequeño, es necesario que tengamos suficientes datos para entrenar. Con esto en mente, usar 20% de los datos para pruebas es mas que suficiente, ya que al no tener muchos datos, si reducimos aun mas test_size, un acierto o fallo va a tener mucha mayor influencia, causando que aciertos aleatorios o fallos pequeños causen un desajuste enorme a la hora de entrenar nuestro modelo y en el peor de los casos, que una serie de aciertos/fallos seguidos causen un sobreajuste/desajuste en el modelo